# NLP Tutorial 2: Named-Entity Recognition for German
**Group 2 (German)** · NLTK + spaCy `de_core_news_md` · corpus from Tutorial 1 (100 Wikipedia introductions)
**Members:** Blevis Allushi · Darsi Ramaj · Geri Belba

This notebook adds a named-entity recogniser to the corpus. It is the step towards the final search engine: every text gets an **entity index** (who, what organisation, which place, which date or amount), with one canonical name and a Wikidata id per entity.

| Group | Tags | How |
|---|---|---|
| ENAMEX | PER, ORG, LOC, GPE | NLTK `CRFTagger` (linear-chain CRF) + knowledge base + context rules |
| NUMEX | MONEY, PERCENT, QUANTITY | NLTK `RegexpParser` chunk grammar |
| TIMEX | DATE, TIME | NLTK `RegexpParser` chunk grammar |

Labels use the **BIO scheme**; nested names get a second BIO layer. All code is in `src/sprache/ner/`; this notebook calls it and discusses the results.

**Run:** Kernel → *Restart Kernel and Run All Cells* (about 3 minutes), or `python -m sprache notebook`. No internet needed: the knowledge base is committed in `data/ner/`.

| Section | Content | Owner |
|---|---|---|
| 0-1 | Setup, tag set, BIO scheme | Blevis |
| 2 | Data: knowledge base, silver labels, gold standard | Blevis |
| 3 | The pipeline on an example | Blevis |
| 4 | Evaluation and error analysis | Blevis |
| 5 | Hard cases: metonymy, ambiguity, variation, nesting, noise | Blevis |
| 6 | Low-resource: learning curve | Blevis |
| 7-8 | Entity index for the search engine, discussion | Blevis |

## 0 · Setup

In [1]:
import sys, time
from pathlib import Path
try:
    import sprache
except ImportError:
    sys.path.insert(0, str(Path('../src').resolve()))
    import sprache

import nltk, spacy, pycrfsuite
import pandas as pd
from IPython.display import Markdown, display

from sprache import analysis, config, corpus
from sprache.ner import bio, context, evaluate, experiments as ex, noise, rules, silver, text
from sprache.ner.doc import Sentence, parse, read_conll
from sprache.ner.kb import KnowledgeBase
from sprache.ner.pipeline import NERPipeline, mentions_table
from sprache.ner.tagger import EntityTagger, MODEL_FILE

pd.set_option('display.max_colwidth', 110)
pd.set_option('display.width', 200)
print('Python  :', sys.version.split()[0])
print('NLTK    :', nltk.__version__)
print('spaCy   :', spacy.__version__)
nlp = analysis.load_model()
print('Model   :', f"{nlp.meta['lang']}_{nlp.meta['name']} {nlp.meta['version']}")
print('sprache :', sprache.__version__)
REBUILD = False      # True: retrain the production model and rewrite data/ner/entities.csv

Python  : 3.11.17
NLTK    : 3.10.4
spaCy   : 3.8.18


Model   : de_core_news_md 3.8.0
sprache : 3.0.0


## 1 · Tag set and BIO scheme

| Tag | Meaning | Example from the corpus |
|---|---|---|
| PER | person | *Johann Sebastian Bach*, *Otto I.* |
| ORG | organisation, institution, company | *Deutscher Bundestag*, *BMW* |
| GPE | geo-political entity: a place **with a government** (country, state, city, district) | *Berlin*, *Freistaat Bayern* |
| LOC | other places: rivers, mountains, regions without a government, buildings | *Rhein*, *Mitteleuropa*, *Berliner Mauer* |
| MONEY, PERCENT, QUANTITY | amounts | *1,2 Milliarden Euro*, *3,5 %*, *891 Quadratkilometern* |
| DATE, TIME | points and periods in time | *21. März 1685*, *Mitte des 18. Jahrhunderts*, *14:30 Uhr* |

**Annotation guidelines** used for the gold standard: titles (*Herzog*, *Kaiser*) and articles are not part of a name; adjectives derived from names (*deutsche*, *Münchner*) are not entities; works, events, languages and movements (*Oktoberfest*, *Sturm und Drang*) are outside the tag set; a number alone is not tagged (it needs a unit, a scale word or a currency); a place that acts as an institution or a team is labelled by its **contextual** type (section 5).

**BIO**: the first token of an entity gets `B-TYPE`, the following ones `I-TYPE`, everything else `O`. BIO is flat, so a name inside a name (*Universität [Wien]*) goes into a second, *inner* layer, as in the GermEval 2014 task. The same labels as an NLTK chunk tree:

In [2]:
words = 'Sie studierte an der Universität Wien bei Johann Sebastian Bach .'.split()
outer = bio.spans_to_bio(len(words), [(4, 6, 'ORG'), (7, 10, 'PER')])
inner = bio.spans_to_bio(len(words), [(5, 6, 'GPE')])
display(pd.DataFrame({'token': words, 'outer BIO': outer, 'inner BIO': inner}).T)
print(bio.to_tree(words, outer))

,0,1,2,3,4,5,6,7,8,9,10
token,Sie,studierte,an,der,Universität,Wien,bei,Johann,Sebastian,Bach,.
outer BIO,O,O,O,O,B-ORG,I-ORG,O,B-PER,I-PER,I-PER,O
inner BIO,O,O,O,O,O,B-GPE,O,O,O,O,O


(S
  Sie/X
  studierte/X
  an/X
  der/X
  (ORG Universität/X Wien/X)
  bei/X
  (PER Johann/X Sebastian/X Bach/X)
  ./X)


## 2 · Data: no hand-labelled training set (low-resource setting)

There is no German NER training data in this project, so the labels come from the encyclopedia itself (**distant supervision**):

1. Every Wikipedia intro links its important names to their articles. `python -m sprache ner-kb` reads the links of the *exact revision* of each of our 100 texts.
2. Wikidata says what each linked article is (*human* → PER, *city* → GPE, *company* → ORG, *river* → LOC). The class hierarchy is walked a few levels up; the type reached on the shortest path wins, so the *Berlin Wall* is a structure (LOC), not an administrative unit.
3. Events, periods, movements, works and concepts (*Hauptstadt*, *Bundesstaat*) are excluded.

In [3]:
kb_df = pd.read_csv(config.NER_KB).fillna('')
links = pd.read_csv(config.NER_LINKS).fillna('')
print(f"{len(links):,} links in 100 intros -> {links.qid.ne('').sum():,} with a Wikidata item -> "
      f"{kb_df.type.ne('').sum():,} typed entities")
display(kb_df[kb_df.type != ''].type.value_counts().rename('entities').to_frame().T)
display(links[links.type != ''].sample(8, random_state=3)[['file', 'anchor', 'target', 'qid', 'type']])

3,719 links in 100 intros -> 3,717 with a Wikidata item -> 1,018 typed entities


type,LOC,GPE,ORG,PER
entities,352,275,244,147


,file,anchor,target,qid,type
1801,text047.txt,Bundesverwaltungsgericht,Bundesverwaltungsgericht (Deutschland),Q702424,ORG
211,text003.txt,Herder,Johann Gottfried Herder,Q155547,PER
1371,text035.txt,Wolfgang Amadeus Mozart,Wolfgang Amadeus Mozart,Q254,PER
1782,text047.txt,Universitäten,Universität Leipzig,Q154804,ORG
457,text011.txt,römisch-katholische,Römisch-katholische Kirche,Q9592,ORG
2069,text053.txt,seiner Kunstsammlungen,Staatliche Kunstsammlungen Dresden,Q653002,LOC
2178,text056.txt,Linke,Die Linke,Q49764,ORG
3157,text081.txt,Klaus,Klaus Mann,Q77087,PER


**Silver labels.** A linked name gets the type of its target and the label is propagated to every later mention in the same text, including variants (genitive *Bachs*, surname alone *Bach*, Wikidata aliases). spaCy's own entities fill the gaps for proper nouns nobody linked. Numbers and dates come from the chunk grammar (section 3).

**Gold standard.** One text per topic (`text001`-`text008`, 85 sentences, 1,945 tokens) was annotated by hand with the guidelines above; it is never used for training. During evaluation the knowledge base only contains entities linked from the 92 training texts, so the system meets the gold texts as new text.

In [4]:
gold = read_conll(config.NER_GOLD)
silver_all = read_conll(config.NER_SILVER)
def counts(sents, layer='outer'):
    return pd.Series([t for s in sents for _, _, t in s.spans(layer)]).value_counts()
display(pd.DataFrame({'silver (100 texts)': counts(silver_all), 'gold (8 texts)': counts(gold),
                      'gold, nested': counts(gold, 'inner')}).reindex(list(config.ENTITY_TYPES)).fillna(0).astype(int).T)

,PER,ORG,LOC,GPE,MONEY,PERCENT,QUANTITY,DATE,TIME
silver (100 texts),217,275,368,695,2,2,77,380,0
gold (8 texts),11,9,50,75,1,1,17,40,0
"gold, nested",0,0,2,5,0,0,0,0,0


## 3 · The pipeline on one example

```
text -> NLTK Punkt + word tokenizer -> [normalisation for noisy text]
     -> spaCy POS, lemmas, dependencies on the NLTK tokens (spaCy helps, NLTK leads)
     -> ENAMEX: NLTK CRFTagger (features: word shape, affixes, POS, gazetteer, spaCy's label, +-2 window)
     -> context layer: type votes, metonymy, one sense per document, lists, nesting, canonical names
     -> NUMEX/TIMEX: NLTK RegexpParser chunk grammar
     -> outer + inner BIO, entity table
```

In [5]:
if REBUILD or not MODEL_FILE.exists():
    t0 = time.time(); ex.build_index(nlp); print(f'trained production model in {time.time() - t0:.0f} s')
kb_full = KnowledgeBase.load()
pipe = NERPipeline(nlp, kb_full, EntityTagger(kb_full).load(),
                   normaliser=ex._normaliser(kb_full, ex.all_files()))
example = ('Johann Sebastian Bach wurde am 21. März 1685 in Eisenach geboren. Ab 1723 wirkte er als '
           'Thomaskantor an der Thomasschule in Leipzig; Bachs Gehalt betrug rund 100 Taler.')
sents = pipe(example)
for s in sents:
    display(pd.DataFrame({'token': s.words, 'POS': s.tags, 'spaCy': s.ents, 'outer': s.outer, 'inner': s.inner}).T)
display(mentions_table(sents).drop(columns=['file']))

,0,1,2,3,4,5,6,7,8,9,10,11
token,Johann,Sebastian,Bach,wurde,am,21.,März,1685,in,Eisenach,geboren,.
POS,NE,NE,NE,VAFIN,APPRART,ADJA,NN,CARD,APPR,NE,VVPP,$.
spaCy,B-PER,I-PER,I-PER,O,O,O,O,O,O,B-LOC,O,O
outer,B-PER,I-PER,I-PER,O,O,B-DATE,I-DATE,I-DATE,O,B-GPE,O,O
inner,O,O,O,O,O,O,O,O,O,O,O,O


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18
token,Ab,1723,wirkte,er,als,Thomaskantor,an,der,Thomasschule,in,Leipzig,;,Bachs,Gehalt,betrug,rund,100,Taler,.
POS,APPR,CARD,VVFIN,PPER,APPR,NN,APPR,ART,NN,APPR,NE,$.,NE,NN,VVFIN,ADV,CARD,NN,$.
spaCy,O,O,O,O,O,O,O,O,B-ORG,O,B-LOC,O,B-PER,O,O,O,O,O,O
outer,O,B-DATE,O,O,O,O,O,O,B-ORG,O,B-GPE,O,B-PER,O,O,O,B-MONEY,I-MONEY,O
inner,O,O,O,O,O,O,O,O,O,O,O,O,O,O,O,O,O,O,O


,sent,start,end,text,type,base_type,reading,layer,canonical,qid
0,0,0,3,Johann Sebastian Bach,PER,PER,literal,outer,Johann Sebastian Bach,Q1339
1,0,5,8,21. März 1685,DATE,DATE,literal,outer,21. März 1685,
2,0,9,10,Eisenach,GPE,GPE,literal,outer,Eisenach,Q7070
3,1,1,2,1723,DATE,DATE,literal,outer,1723,
4,1,8,9,Thomasschule,ORG,ORG,literal,outer,Thomasschule,
5,1,10,11,Leipzig,GPE,GPE,literal,outer,Leipzig,Q2079
6,1,12,13,Bachs,PER,PER,literal,outer,Johann Sebastian Bach,Q1339
7,1,16,18,100 Taler,MONEY,MONEY,literal,outer,100 Taler,


NUMEX and TIMEX come from a cascaded NLTK chunk grammar over word classes (YEAR, ORD, MONTH, CUR, UNIT, ...). The order of the stages resolves ambiguity: *2000 Euro* is MONEY because MONEY is chunked before DATE; *1500 Einwohner* is not a year.

In [6]:
print(rules.GRAMMAR)
w = 'Der Umsatz stieg 2019 um 3,5 % auf 12,4 Mrd. Euro ; dort leben 1500 Einwohner seit Mitte des 18. Jahrhunderts .'.split()
print(rules.chunk(w))


TIME:     {<CLOCK><UHR>?}
          {<NUM|ORD><UHR>}
MONEY:    {<NUM|YEAR>(<DASH><NUM|YEAR>)?<SCALE>?<CUR>}
          {<CUR><NUM|YEAR><SCALE>?}
PERCENT:  {<NUM|YEAR>(<DASH><NUM|YEAR>)?<PCT>}
QUANTITY: {<NUM|YEAR>(<DASH><NUM|YEAR>)?<SCALE>?<UNIT>}
          {<NUM|YEAR><CNT><PRO><UNIT>}
          {<NUM|YEAR|NUMW>(<DASH><NUM|YEAR>)?<SCALE>}
DATE:     {(<MOD><DET>?)?<TADJ>?<ORD>(<DASH|CONJ><TADJ>?<ORD>)?<CENT>}
          {<WEEKDAY>?<MOD>?<ORD>?<MONTH><YEAR>?}
          {<DECADE><DECN>?}
          {<YEAR><DASH><YEAR>}
          {<YEAR>}

(S
  Der/O
  Umsatz/O
  stieg/O
  (DATE 2019/YEAR)
  um/O
  (PERCENT 3,5/NUM %/PCT)
  auf/O
  (MONEY 12,4/NUM Mrd./SCALE Euro/CUR)
  ;/O
  dort/O
  leben/O
  1500/NUM
  Einwohner/CNT
  seit/O
  (DATE Mitte/MOD des/DET 18./ORD Jahrhunderts/CENT)
  ./O)


## 4 · Evaluation on the gold texts
Entity-level, CoNLL-style: an entity counts only if start, end and type are all correct. Each line adds one component.

In [7]:
t0 = time.time()
systems, info = ex.compare_systems(nlp)
print(f'{time.time() - t0:.0f} s')
display(systems)
full = info['tables']['Full pipeline (CRF + context layer + rules)']
display(full)

60 s


,ENAMEX F1,NUMEX F1,TIMEX F1,all F1
system,,,,
spaCy de_core_news_md (as is),0.351,0.000,0.0,0.291
spaCy + KB types + rules,0.570,0.947,1.0,0.689
Gazetteer + rules (no learning),0.631,0.947,1.0,0.752
"CRF, word features only + rules",0.636,0.947,1.0,0.739
CRF + gazetteer + rules,0.693,0.947,1.0,0.778
CRF + gazetteer + spaCy feature + rules,0.705,0.947,1.0,0.789
Full pipeline (CRF + context layer + rules),0.797,0.947,1.0,0.853


,precision,recall,f1,support
type,,,,
PER,0.692,0.818,0.750,11
ORG,0.667,0.222,0.333,9
LOC,0.850,0.680,0.756,50
GPE,0.867,0.867,0.867,75
MONEY,1.000,1.000,1.000,1
PERCENT,1.000,1.000,1.000,1
QUANTITY,0.941,0.941,0.941,17
DATE,1.000,1.000,1.000,40
TIME,0.000,0.000,0.000,0


In [8]:
s = systems['all F1']
display(Markdown(
    f"**Reading the table.** spaCy alone reaches F1 {s.iloc[0]:.2f}: it has no GPE tag and no NUMEX/TIMEX at all, "
    f"and it labels names as PER/ORG/LOC/MISC. Our full pipeline reaches **{s.iloc[-1]:.2f}** "
    f"(ENAMEX {systems['ENAMEX F1'].iloc[-1]:.2f}, NUMEX {systems['NUMEX F1'].iloc[-1]:.2f}, TIMEX {systems['TIMEX F1'].iloc[-1]:.2f}). "
    f"The gazetteer and spaCy features each help the CRF, and the context layer adds "
    f"{systems['ENAMEX F1'].iloc[-1] - systems['ENAMEX F1'].iloc[-2]:+.2f} ENAMEX F1 on top. "
    f"Nested entities (inner layer): F1 {info['tables']['inner layer'].loc['all (micro)', 'f1']:.2f} on "
    f"{int(info['tables']['inner layer'].loc['all (micro)', 'support'])} gold cases."))

**Reading the table.** spaCy alone reaches F1 0.29: it has no GPE tag and no NUMEX/TIMEX at all, and it labels names as PER/ORG/LOC/MISC. Our full pipeline reaches **0.85** (ENAMEX 0.80, NUMEX 0.95, TIMEX 1.00). The gazetteer and spaCy features each help the CRF, and the context layer adds +0.09 ENAMEX F1 on top. Nested entities (inner layer): F1 0.83 on 7 gold cases.

**Error analysis**: every miss and every false alarm of the full pipeline.

In [9]:
g, f = info['gold'], info['full']
errs = evaluate.errors([s.outer for s in g], [s.outer for s in f], [s.words for s in g])
print(len(errs), 'errors')
display(errs)

45 errors


,sentence,gold,gold_type,predicted
0,8,Bundesrepublik,GPE,-
1,10,Bundesregierung,ORG,-
2,10,Bundesrates,ORG,-
3,16,Rhenus,LOC,-
4,16,Reno,LOC,Reno/GPE
5,16,Rein,LOC,Rein/PER
6,16,Rein/Ragn,LOC,-
7,16,Rhy/Ry,LOC,-
8,16,Rhing,LOC,-
9,16,West-,LOC,-


The errors fall into a few groups:
* **Names the knowledge base cannot know**: historical states (*Ostfrankenreich*, *Regnum Teutonicum*), foreign names of the Rhine (*Rhenus*, *Rhing*), river arms (*Lek*). They were only linked from the gold text itself, which the evaluation hides on purpose.
* **GPE vs LOC**: river names the model has never seen get the more frequent type GPE (*Waal*, *IJssel*); *Lothringen* and *Deutschschweiz* sit on the guideline boundary (region with or without government).
* **ORG with adjective + common noun** (*Bayerischen Staatsregierung*, *Bundesrates*): every German noun is capitalised, so these look like ordinary noun phrases.
* **Common nouns as names** (*Einzugsgebiet*, *Staaten*): capitalisation is no signal in German.

## 5 · Hard cases

The gold texts are encyclopedic and contain almost no metonymy, so a hand-written **challenge set** (`data/ner/challenge.txt`) tests each phenomenon directly. How each one is handled:

| Phenomenon | Example | Mechanism |
|---|---|---|
| Metonymy | *Berlin lehnt den Vorschlag ab* (Berlin = government) | spaCy dependency parse: a place that is the subject of an agentive verb (*ablehnen, fordern, verhandeln*) becomes ORG, reading *place-for-institution*; subject/object/*gegen* of a sports verb becomes *place-for-team*; a company as object of *fahren/bauen* with an article is a product, not an entity |
| Type ambiguity | *Bosch* (person or company), *August* (month or name) | votes from CRF, knowledge base and cue words (*Herzog*, *GmbH*, *bei*, head nouns such as *Universität*); one type per name per text; members of one list share a type; dates need a temporal cue |
| Variation | *Bach / Bachs / Johann Sebastian Bach*, *BRD*, *Muenchen* | genitive stripping, umlaut-insensitive matching, inflection-insensitive matching, Wikidata aliases, surname linked to the full name in the same text, canonical name + Wikidata id |
| Nested | *Universität [Wien]*, *Landkreis [München]*, *Frankfurt am [Main]* | second BIO layer; head nouns (*Landkreis*, *Universität*) build the outer name; KB names inside a name |
| Noisy | *ich war in muenchen und koeln*, typos | normaliser: case and umlaut restoration from a lexicon of the corpus + KB, typo repair within edit distance 1 (`nltk.edit_distance`) |
| Low-resource | no labelled data | distant supervision, gazetteer dropout, rules for closed classes (section 6) |

In [10]:
norm = ex._normaliser(info['kb'], ex.train_files())
ch = ex.challenge(nlp, info['tagger'], info['kb'], norm)
summary = ch.groupby('phenomenon')[['correct', 'expected_n', 'spurious']].sum()
summary['recall'] = (summary.correct / summary.expected_n).round(2)
display(summary)
display(ch[['phenomenon', 'sentence', 'expected', 'found']])

,correct,expected_n,spurious,recall
phenomenon,,,,
ambiguity,15,16,1,0.94
literal,4,5,1,0.80
metonymy,10,11,1,0.91
nested,9,10,0,0.90
noisy,10,11,0,0.91
numex,6,6,0,1.00
timex,4,4,0,1.00
variation,9,10,1,0.90


,phenomenon,sentence,expected,found
0,metonymy,Berlin lehnt den Vorschlag ab .,Berlin/ORG,Berlin/ORG
1,metonymy,Washington und Moskau verhandeln über Abrüstung .,Washington/ORG; Moskau/ORG,Washington/ORG; Moskau/ORG
2,metonymy,Brüssel fordert strengere Regeln für die Banken .,Brüssel/ORG,Brüssel/ORG
3,metonymy,Deutschland gewann das Finale gegen Brasilien .,Deutschland/ORG; Brasilien/ORG,Deutschland/ORG; Brasilien/ORG
4,metonymy,Der FC Bayern München besiegte Dortmund am Samstag .,FC Bayern München/ORG; München/GPE; Dortmund/ORG,FC Bayern München/ORG; Bayern/GPE; München/GPE; Dortmund/ORG
5,metonymy,Die Bundesregierung in Berlin hat die Sanktionen beschlossen .,Bundesregierung/ORG; Berlin/GPE,Berlin/GPE
6,literal,Wir fahren morgen nach Berlin .,Berlin/GPE,Berlin/GPE
7,literal,Paris ist die Hauptstadt von Frankreich .,Paris/GPE; Frankreich/GPE,Paris/GPE; Frankreich/GPE
8,literal,Brasilien ist das größte Land Südamerikas .,Brasilien/GPE; Südamerikas/LOC,Brasilien/GPE; Südamerikas/GPE
9,ambiguity,Robert Bosch gründete 1886 in Stuttgart eine Werkstatt .,Robert Bosch/PER; 1886/DATE; Stuttgart/GPE,Robert Bosch/PER; 1886/DATE; Stuttgart/GPE


**Noisy text.** The gold texts are corrupted token by token (so the gold labels still fit) and tagged with and without normalisation:

In [11]:
nz = ex.noise_robustness(nlp, info['tagger'], info['kb'])
display(nz)
display(Markdown(
    f"Lower-casing alone drops F1 from {nz.loc['clean', 'F1 as is']:.2f} to {nz.loc['lower', 'F1 as is']:.2f}: "
    f"in German the capital letter of a name is shared with every noun, but losing it still removes the main cue. "
    f"Social-media style (lower case + ae/oe/ue + typos) reaches only {nz.loc['social', 'F1 as is']:.2f}. "
    f"Normalisation brings it back to **{nz.loc['social', 'F1 normalised']:.2f}**. On clean text it costs "
    f"{nz.loc['clean', 'F1 as is'] - nz.loc['clean', 'F1 normalised']:.3f} F1, so it is switched on only for noisy input."))
for kind in noise.KINDS:
    print(f'{kind:7s}', ' '.join(noise.corrupt(g[0].words, kind, seed=0)))

,F1 as is,F1 normalised
text,,
clean,0.853,0.848
lower,0.529,0.834
ascii,0.827,0.846
typo,0.790,0.835
social,0.458,0.822


Lower-casing alone drops F1 from 0.85 to 0.53: in German the capital letter of a name is shared with every noun, but losing it still removes the main cue. Social-media style (lower case + ae/oe/ue + typos) reaches only 0.46. Normalisation brings it back to **0.82**. On clean text it costs 0.005 F1, so it is switched on only for noisy input.

lower   berlin ist die hauptstadt und ein land der bundesrepublik deutschland .
ascii   Berlin ist die Hauptstadt und ein Land der Bundesrepublik Deutschland .
typo    Berlin ist die Hauptstadt und ein Land der Bundesrepublik Deutschland .
social  berlin ist die hauptstadt und ein land der bundesrepublik deutschland .


## 6 · Low-resource: how much silver data is needed?
The CRF is trained on 10 % to 100 % of the 92 training texts (all labels automatic) and tested on the gold texts.

In [12]:
lc = ex.learning_curve(nlp)
display(lc)
display(Markdown(
    f"With only {int(lc.texts.iloc[0])} texts ({int(lc.tokens.iloc[0]):,} tokens) the full system already reaches "
    f"ENAMEX F1 {lc['CRF + context'].iloc[0]:.2f}; all {int(lc.texts.iloc[-1])} texts give {lc['CRF + context'].iloc[-1]:.2f}. "
    f"The curve is flat because the silver labels are noisy (more of them also means more of their errors) and because "
    f"the knowledge base and the context rules do much of the work. The context layer adds about "
    f"{(lc['CRF + context'] - lc['CRF only']).mean():.2f} F1 at every size."))

context,fraction,texts,tokens,CRF only,CRF + context
0,0.10,9,2230,0.700,0.797
1,0.25,23,5925,0.667,0.786
2,0.50,46,10805,0.719,0.789
3,1.00,92,22010,0.705,0.797


With only 9 texts (2,230 tokens) the full system already reaches ENAMEX F1 0.80; all 92 texts give 0.80. The curve is flat because the silver labels are noisy (more of them also means more of their errors) and because the knowledge base and the context rules do much of the work. The context layer adds about 0.09 F1 at every size.

## 7 · Entity index for the search engine
`python -m sprache ner` tags the whole corpus with the production model (trained on all 100 texts) and writes `data/ner/entities.csv`: one row per mention with type, reading, layer, canonical name and Wikidata id.

In [13]:
ents = pd.read_csv(config.NER_OUTPUT).fillna('')
print(f"{len(ents):,} mentions in {ents.file.nunique()} texts; {ents.qid.ne('').mean():.0%} linked to Wikidata")
display(ents.type.value_counts().to_frame().T)
top = (ents[ents.type.isin(config.ENAMEX)].groupby(['canonical', 'type', 'qid']).file.nunique()
       .sort_values(ascending=False).rename('texts').head(12).reset_index())
display(top)

2,066 mentions in 94 texts; 68% linked to Wikidata


type,GPE,DATE,LOC,ORG,PER,QUANTITY,PERCENT,MONEY
count,743,380,378,269,215,77,2,2


,canonical,type,qid,texts
0,Deutschland,GPE,Q183,47
1,Österreich,GPE,Q40,15
2,Europa,LOC,Q46,14
3,Europäische Union,GPE,Q458,11
4,Schweiz,GPE,Q39,10
5,Berlin,GPE,Q64,10
6,Rhein,LOC,Q584,8
7,Preußen,GPE,Q38872,8
8,Mitteleuropa,LOC,Q27509,7
9,Frankreich,GPE,Q142,7


A first search-engine query: which texts mention an entity, whatever spelling they use? The Wikidata id joins all variants.

In [14]:
from sprache.ner.kb import fold

def search(query):
    # texts that mention an entity: by Wikidata id (all spellings) or by a word of its canonical name
    key = fold(query)
    qids = {e.qid for e in kb_full.candidates(query)}
    hit = ents.qid.isin(qids) | ents.canonical.map(lambda c: key == fold(c) or key in fold(c).split())
    rows = ents[hit & ents.type.isin(config.ENAMEX)]
    src = pd.DataFrame(corpus.read_sources())[['file', 'title']]
    return (rows.groupby('file').agg(mentions=('text', 'count'),
                                     entity=('canonical', lambda x: ', '.join(sorted(set(x)))),
                                     forms=('text', lambda x: ', '.join(sorted(set(x)))))
            .reset_index().merge(src, on='file').sort_values('mentions', ascending=False))

display(Markdown('**Query: `Bach`**'))
display(search('Bach'))
display(Markdown('**Query: `muenchen`** (noisy spelling)'))
display(search('muenchen').head(8))

**Query: `Bach`**

,file,mentions,entity,forms,title
0,text028.txt,9,"Bach, Johann Sebastian Bach, Musikerfamilie Bach","Bach, Bachs, Johann Sebastian Bach, Musikerfamilie Bach",Johann Sebastian Bach
1,text047.txt,1,Johann Sebastian Bach,Johann Sebastian Bach,Leipzig
2,text084.txt,1,Johann Sebastian Bach,Johann Sebastian Bach,Weimar


**Query: `muenchen`** (noisy spelling)

,file,mentions,entity,forms,title
1,text008.txt,10,"Landkreises München, Metropolregion München, München, Planungsregion München","Landkreises München, Metropolregion München, München, Planungsregion München",München
0,text005.txt,2,"Landeshauptstadt München, München","München, Stadtverwaltung München",Oktoberfest
4,text035.txt,2,München,München,Wolfgang Amadeus Mozart
2,text010.txt,1,München,München,Siemens
3,text018.txt,1,München,München,BMW
5,text077.txt,1,München,München,Max-Planck-Gesellschaft


## 8 · Discussion

**What works.** Numbers and dates are nearly solved by an NLTK chunk grammar (closed classes need no learning). Names reach F1 around 0.8 without a single hand-labelled training sentence: Wikipedia links and Wikidata replace annotators, spaCy fills gaps, and an NLTK CRF learns to generalise from both. Linking every mention to a Wikidata id turns name variation into a non-problem for search.

**What is hard in German.** Capitalisation marks every noun, so it is a weak name signal (*Einzugsgebiet* tagged LOC). Inflection changes names (*Europäischen Union*, *Bachs*, *Niederlanden*). GPE vs LOC depends on whether a region has a government, which the text itself rarely says.

**Limits.**
* The gold standard is small: 8 texts, 204 entities, one annotator. Differences of 1-2 points are within noise. A second annotator and an agreement score would be the next step.
* Metonymy and noisy text are measured on our own challenge set and on synthetic noise, not on real tweets.
* The silver labels inherit Wikidata's view of the world: the EU is a GPE, *Metropolregion* a GPE there but LOC in our guidelines.
* The metonymy rules need a correct parse; the spaCy parser makes errors on long sentences.

**Next steps for the search engine:** index the cleaned texts with the entity table as facets (type, Wikidata id); expand queries with Wikidata aliases; rank by entity frequency per text.

### Deliverables
- [x] ENAMEX (PER, ORG, LOC, GPE), NUMEX (MONEY, PERCENT, QUANTITY), TIMEX (DATE, TIME)
- [x] BIO scheme, with a second layer for nested entities (`data/ner/silver.conll`, `gold.conll`)
- [x] Implemented with NLTK (Punkt, `RegexpParser`, `CRFTagger`, chunk trees, `edit_distance`), spaCy as a helper (POS, parse, second opinion)
- [x] Metonymy, type ambiguity, variation, nested, noisy and low-resource cases handled and measured (sections 5-6)
- [x] Evaluation against a hand-checked gold standard, with baselines and error analysis (section 4)
- [x] Entity index for the search engine (`data/ner/entities.csv`, section 7)